# 13.1 Running scripts: include and commands

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

AMPL provides several commands that cause input to be taken from a file. The command

```ampl
include filename
```

is replaced by the contents of the named file. An include can even appear in the middle
of some other statement, and does not require a terminating semicolon.


The `model` and `data` commands that appear in most of our examples are special
cases of include that put the command interpreter into `model` or `data` mode before
reading the specified file. By contrast, include leaves the mode unchanged. To keep
things simple, the examples in this book always assume that `model` reads a file of `model`
declarations, and that `data` reads a file of `data` values. You may use any of `model`,
`data` and include to read any file, however; the only difference is the mode that is set
when reading starts. Working with a small `model`, for example, you might find it convenient
to store in one file all the `model` declarations, a `data` command, and all the `data`
statements; either a `model` or an include command could read this file to set up both
`model` and `data` in one operation.

As an illustration, if the file dietu.run contains

```ampl
model dietu.mod;
data dietu.dat;
solve;
option display_1col 5;
option display_round 1;
display Buy;
```

then including it will load the `model` and `data`, run the problem, and `display` the optimal
values of the variables:

```ampl
ampl: include dietu.run;
MINOS 5.5: optimal solution found.

6 iterations, objective 74.27382022

Buy [*] :=
BEEF 2.0         FISH    2.0       MCH    2.0       SPG    5.3
       CHK 10.0        HAM    2.0       MTL    6.2       TUR    2.0
;
```

When an included file itself contains an include, `model` or `data` command, reading
of the first file is suspended while the contents of the contained file are included. In this
example, the command include dietu.run causes the subsequent inclusion of the
files dietu.mod and dietu.dat.

One particularly useful kind of include file contains a list of option commands
that you want to run before any other commands, to modify the default options. You can
arrange to include such a file automatically at startup; you can even have AMPL write
such a file automatically at the end of a session, so that your option settings will be
restored the next time around. Details of this arrangement depend on your operating
system; see [Sections A.14.1 and A.23](#missing) <!--- xTODO missing reference --->.

The statement

```ampl
commands filename ;
```

is very similar to include, but is a true statement that needs a terminating semicolon
and can only appear in a context where a statement is legal.

To illustrate commands, consider how we might perform a simple sensitivity analysis
on the multi-period production problem of [Section 4.2](../04/4_2_a_multiperiod_production_model.ipynb). Only 32 hours of production
time are available in week 3, compared to 40 hours in the other weeks. Suppose that we
want to see how much extra profit could be gained for each extra hour in week 3. We can
accomplish this by repeatedly solving, displaying the solution values, and increasing
avail[3]:

```ampl
ampl: model steelT.mod;
ampl: data steelT.dat;
ampl: solve;
MINOS 5.5: optimal solution found.

15 iterations, objective 515033
ampl: display Total_Profit >steelT.sens;
ampl: option display_1col 0;
ampl: option omit_zero_rows 0;
ampl: display Make >steelT.sens;
ampl: display Sell >steelT.sens;
ampl: option display_1col 20;
ampl: option omit_zero_rows 1;
ampl: display Inv >steelT.sens;
ampl: let avail[3] := avail[3] + 5;
ampl: solve;
MINOS 5.5: optimal solution found.

1 iterations, objective 532033
ampl: display Total_Profit >steelT.sens;
ampl: option display_1col 0;
ampl: option omit_zero_rows 0;
ampl: display Make >steelT.sens;
ampl: display Sell >steelT.sens;
ampl: option display_1col 20;
ampl: option omit_zero_rows 1;
ampl: display Inv >steelT.sens;
ampl: let avail[3] := avail[3] + 5;
ampl: solve;
MINOS 5.5: optimal solution found.

1 iterations, objective 549033
ampl:
```

To continue trying values of avail[3] in steps of 5 up to say 62, we must complete
another four `solve` cycles in the same way. We can avoid having to type the same commands
over and over by creating a new file containing the commands to be repeated:

```ampl
solve;
display Total_Profit >steelT.sens;
option display_1col 0;
option omit_zero_rows 0;
display Make >steelT.sens;
display Sell >steelT.sens;
option display_1col 20;
option omit_zero_rows 1;
display Inv >steelT.sens;
let avail[3] := avail[3] + 5;
```

If we call this file steelT.sa1, we can execute all the commands in it by typing the
single line commands steelT.sa1:

```ampl
ampl: model steelT.mod;
ampl: data steelT.dat;
ampl: commands steelT.sa1;
MINOS 5.5: optimal solution found.

15 iterations, objective 515033
ampl: commands steelT.sa1;
MINOS 5.5: optimal solution found.

1 iterations, objective 532033
ampl: commands steelT.sa1;
MINOS 5.5: optimal solution found.

1 iterations, objective 549033
ampl: commands steelT.sa1;
MINOS 5.5: optimal solution found.

2 iterations, objective 565193
ampl:
```

(All output from the `display` command is redirected to the file steelT.sens,
although we could just as well have made it appear on the screen.)
In this and many other cases, you can substitute include for commands. In general
it is best to use commands within command scripts, however, to avoid unexpected
interactions with repeat, for and if statements.